In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plot

from RNN import logits
%matplotlib inline

torch.manual_seed(12046)

In [3]:
dataset = datasets.MNIST(root='./mnist', train=True, download=True, transform=transforms.ToTensor())
train_set, val_set = random_split(dataset, [50000, 10000])
test_set = datasets.MNIST(root='./mnist', train=False, download=True, transform=transforms.ToTensor())
len(train_set),len(val_set),len(test_set)

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.63MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 134kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 895kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.53MB/s]


(50000, 10000, 10000)

In [4]:
train_loader = DataLoader(train_set, batch_size=500, shuffle=True)
val_loader = DataLoader(val_set, batch_size=500, shuffle=True)
test_loader = DataLoader(test_set, batch_size=500, shuffle=True)

In [5]:
eval_iters = 10

def estimate_loss(model):
    re = {}
    model.eval()
    re['train'] = _loss(model, train_loader)
    re['val'] = _loss(model, val_loader)
    re['test'] = _loss(model, test_loader)
    model.train()
    return re

@torch.no_grad()
def _loss(model, dataloader):
    '''
    B (Batch Size)：批次大小，这里为500
    C (Channels)：通道数，这里为1（灰度图）
    H (Height)：图像高度，这里为28
    W (Width)：图像宽度，这里为28
    '''
    loss = []
    acc = []
    data_iter = iter(dataloader) #将其转换为迭代器
    for t in range(eval_iters):
        inputs, labels = next(data_iter)
        # inputs: (500, 1, 28, 28)
        # labels: (500)
        B, C, H, W = inputs.shape
        logits = model(inputs)
        loss.append(F.cross_entropy(logits, labels))
        preds = torch.argmax(logits, dim=1)
        acc.apend((preds == labels).sum()/B)
        re = {
            'loss':torch.tensor(loss).mean().item(),
            'acc':torch.tensor(acc).mean().item(),
        }
        return re

def train_model(model, optimizer,epochs=10, penalty=False, l1_lambda=0.001, l2_lambda=0.002):
    lossi = []
    for e in range(epochs):
        for data in train_loader:
            inputs, labels = data
            #B,,C,H,W = inputs.shape
            logits = model(inputs)
            loss = F.cross_entropy(logits, labels)
            # 添加正择项
            if penalty:
                w = torch.cat([p.view(-1) for p in model.parameters()])
                l1_penalty = l1_lambda * w.abs().sum()
                l2_penalty = l2_lambda * w.square().sum()
                loss = loss + l1_penalty + l2_penalty
            lossi.append(loss.item())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        stats = estimate_loss(model)
        train_loss = f'{stats["train"]['losss']:.3f}'
        val_loss = f'{stats["val"]['loss']:.3f}'
        test_loss = f'{stats["test"]['loss']:.3f}'
        print(f'epoch {e} train {train_loss} val {val_loss} test {test_loss}')
    return lossi

In [6]:
# 卷积神经网络中保持空间维度不变的核心技巧：使input和output形状一样
channels = torch.randint(1,10,(1,))
conv1 = nn.Conv2d(channels, channels, (3,3), stride=1, padding=1)
x= torch.randn(1, channels, 28, 28)
x.shape, conv1(x).shape

(torch.Size([1, 4, 28, 28]), torch.Size([1, 4, 28, 28]))

In [7]:
class ResidualBlockSimplified(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, (3,3), stride=1, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, (3,3), stride=1, padding=1)

    def forward(self,x):
        inputs = x
        x = F.relu(self.conv1(x))
        x =self.conv2(x)
        #残差连接
        out = x + inputs
        out = F.relu(out)
        return out

In [8]:
  # 测试
model = ResidualBlockSimplified(3)
x = torch.randn(1, 3, 28, 28)
model(x).shape

torch.Size([1, 3, 28, 28])

In [48]:
# 这两个卷积操作输出是一样的
stride = torch.randint(1, 10,(1,))
in_channels = torch.randint(1, 10,(1,))
out_channels = torch.randint(1, 10,(1,))
conv1 = nn.Conv2d(in_channels, out_channels, (3,3), stride=stride, padding=1)
conv2 = nn.Conv2d(in_channels, out_channels, (1,1), stride=stride, padding=0)
x = torch.randn(1, in_channels, 28, 28)
conv1(x).shape, conv2(x).shape

(torch.Size([1, 9, 6, 6]), torch.Size([1, 9, 6, 6]))

In [76]:
class ResidualBlock(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, (3,3), stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, (3,3), stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = None
        self.bn3 = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Conv2d(in_channels, out_channels, (1,1), stride=stride, padding=0)
            self.bn3 = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        inputs = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        if self.downsample is not None:
            inputs = self.bn3(self.downsample(inputs))
        out = x + inputs
        out = F.relu(out)
        return out

In [ ]:
"""
# 非经典残差连接
class ResidualBlock(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, (3,3), stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, (3,3), stride=1, padding=1)
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Conv2d(in_channels, out_channels, (1,1), stride=stride, padding=0)
        self.bn2 = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        inputs = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.conv2(x)
        if self.downsample is not None:
            inputs = self.downsample(inputs)
        out = self.bn2(x + inputs)
        out = F.relu(out)
        return out
"""

In [74]:
class ResNet(nn.Module):

    def __init__(self):
        super().__init__()
        self.block1 = ResidualBlock(1, 20)
        self.block2 = ResidualBlock(20, 40, stride=2)
        self.block3 = ResidualBlock(40, 60, stride=2)
        self.block4 = ResidualBlock(60, 80, stride=2)
        self.block5 = ResidualBlock(80, 100, stride=2)
        self.block6 = ResidualBlock(100, 120, stride=2)
        self.fc = nn.Linear(120, 10)

    def forward(self, x):
        # x:(B,1,28,28)
        B = x.shape[0]
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.block6(x)
        x = self.fc(x.view(B,-1))
        return x

In [77]:
model = ResNet()
x = torch.randn(10,1,28,28)
model(x).shape

torch.Size([10, 10])